# Example queries: `helpers` (resstock_oedi)

Auto-generated from `tests/query_snapshots/helpers.json`. Each cell
runs one entry from the snapshot suite. Regenerate by running the
matching test with `--update-snapshot` or `--overwrite-snapshot`.


In [1]:
from pathlib import Path
from buildstock_query import BuildStockQuery
import pandas as pd


## Construct the BuildStockQuery object

`cache_folder` points at the snapshot test cache directory so this
notebook reuses parquets that the test suite has already downloaded
from Athena. Queries that are already cached return immediately;
anything new still hits Athena.


In [2]:
# This notebook lives in `tests/example_notebooks/`; the snapshot test
# cache is its sibling `tests/query_snapshots/resstock_oedi_cache/`. Resolve
# the path relative to the notebook directory (`_dh[0]` is set by
# IPython at kernel startup; falls back to CWD outside Jupyter).
_NB_DIR = Path(_dh[0] if "_dh" in globals() else ".").resolve()
_CACHE = (_NB_DIR / "../query_snapshots/resstock_oedi_cache").resolve()
bsq = BuildStockQuery(
    "rescore",
    "buildstock_sdr",
    "resstock_2024_amy2018_release_2",
    buildstock_type="resstock",
    db_schema="resstock_oedi_vu",
    skip_reports=True,
    cache_folder=str(_CACHE),
)


INFO:buildstock_query.query_core:Loading resstock_2024_amy2018_release_2 ...


INFO:botocore.tokens:Loading cached SSO token for nrel-sso


## `upgrade_names_oedi`

get_upgrade_names — was broken before 2026-04-26 fix on two counts: (1) f-string interpolation of self.up_table (a Subquery) produced malformed `FROM SELECT * FROM ...` SQL with no enclosing parens; (2) the method assumed an `apply_upgrade.upgrade_name` column that doesn't exist on OEDI schemas. Fixed to use SA construction and gracefully degrade to `CAST(NULL AS VARCHAR) AS upgrade_name` when the name column is absent — so the dict shape stays consistent across schemas.


In [3]:
result = bsq.get_upgrade_names()
result.head() if hasattr(result, 'head') else result


{1: nan,
 2: nan,
 3: nan,
 4: nan,
 5: nan,
 6: nan,
 7: nan,
 8: nan,
 9: nan,
 10: nan,
 11: nan,
 12: nan,
 13: nan,
 14: nan,
 15: nan,
 16: nan}

## `upgrades_csv_three_buildings_upgrade1`

get_upgrades_csv for upgrade=1 restricted to a tiny known building_id list. Mirrors results_csv_three_buildings on the upgrade-table side.


In [4]:
result = bsq.get_upgrades_csv(upgrade_id='1', restrict=[('bldg_id', [1, 2, 3])])
result.head() if hasattr(result, 'head') else result


INFO:buildstock_query.main:Making results_csv query for upgrade ...


KeyError: "None of ['bldg_id'] are in the columns"

## `buildings_by_locations_state`

get_buildings_by_locations: list bs-keys whose location_col=$STATE_COL_BASELINE is in ['CO']. Different code path from restrict — uses bs_key_cols projection and IN-list filter directly.


In [5]:
result = bsq.get_buildings_by_locations(location_col='in.state', locations=['CO'])
result.head() if hasattr(result, 'head') else result


,bldg_id
0,13
1,16
2,33
3,128
4,182


## `results_csv_three_buildings`

get_results_csv restricted to a tiny known building_id list. Pins the SELECT-* shape of the baseline metadata projection used by basic_usage.ipynb. Bounded to 3 buildings so the data parquet stays small (results_csv has ~150 columns, full state would be 100+ MB).


In [6]:
result = bsq.get_results_csv(restrict=[('bldg_id', [1, 2, 3])])
result.head() if hasattr(result, 'head') else result


INFO:buildstock_query.main:Making results_csv query ...


KeyError: "None of ['bldg_id'] are in the columns"

## `distinct_vals_state_baseline`

get_distinct_vals on the baseline-table state column. Pins the simple SELECT DISTINCT shape — used by notebooks to enumerate categorical values before building restricts. The column name differs by schema (resstock=`in.state`, comstock=`state`); resolved via $STATE_COL_BASELINE.


In [7]:
result = bsq.get_distinct_vals(column='in.state', table_name='resstock_2024_amy2018_release_2_metadata')
result.head() if hasattr(result, 'head') else result


0    CT
1    CO
2    WI
3    MS
4    ID
Name: in.state, dtype: str

## `distinct_count_state_baseline`

get_distinct_count on the baseline-table state column. Pins the COUNT-with-weighted-sum shape (sample_count + weighted_count per category).


In [8]:
result = bsq.get_distinct_count(column='in.state')
result.head() if hasattr(result, 'head') else result


,in.state,sample_count,weighted_count
0,AL,9120,2.300991e+06
1,AR,5536,1.396742e+06
2,AZ,12023,3.033423e+06
3,CA,57394,1.448060e+07
4,CO,9425,2.377943e+06
